# 🧠 MS Lesion Segmentation: Training Notebook
### Stage 1: Infrastructure & Data Ingestion

This notebook is designed to download raw MS MRI datasets and prepare the environment for 3D Deep Learning.

## 🛠 1.1 Environment Setup
Install core medical AI libraries and mount Google Drive for saving model weights.

In [ ]:
!pip install -q monai[all] SimpleITK nibabel tqdm

from google.colab import drive
import os

drive.mount('/content/drive')

# Define Workspace Paths
BASE_DIR = "/content"
DATA_DIR = os.path.join(BASE_DIR, "data/raw")
PROCESSED_DIR = os.path.join(BASE_DIR, "data/processed")
MODEL_SAVE_PATH = "/content/drive/MyDrive/MS_Project/models"

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)

print(f"✅ Workspace initialized. Models will be saved to: {MODEL_SAVE_PATH}")

## 📊 1.2 Data Ingestion
Paste your dataset download links below. The notebook will automatically download and organize them.

In [ ]:
# Dataset A: MSLesSeg (Training Source)
MSLESSEG_URL = "YOUR_LINK_HERE"
if MSLESSEG_URL != "YOUR_LINK_HERE":
    !wget -O mslesseg.zip "{MSLESSEG_URL}"
    !unzip -q mslesseg.zip -d {DATA_DIR}/MSLesSeg
    print("✅ MSLesSeg Downloaded.")

# Dataset B: Mendeley (External Validation)
MENDELEY_URL = "YOUR_LINK_HERE"
if MENDELEY_URL != "YOUR_LINK_HERE":
    !wget -O mendeley.zip "{MENDELEY_URL}"
    !unzip -q mendeley.zip -d {DATA_DIR}/Mendeley
    print("✅ Mendeley Downloaded.")

# Dataset C: Long-MR-MS (Longitudinal Analysis)
LONG_MRMS_URL = "YOUR_LINK_HERE"
if LONG_MRMS_URL != "YOUR_LINK_HERE":
    !wget -O long_mrms.zip "{LONG_MRMS_URL}"
    !unzip -q long_mrms.zip -d {DATA_DIR}/Long-MR-MS
    print("✅ Long-MR-MS Downloaded.")

## 🧠 1.3 The Preprocessing Engine
This cell implements the 6-step medical imaging sequence: Re-orientation, Resampling, Skull-stripping, N4, and Normalization.

In [ ]:
import SimpleITK as sitk
import numpy as np

def preprocess_volume(img_path, target_spacing=(1.0, 1.0, 1.0), is_mask=False):
    # 1. Load
    img = sitk.ReadImage(img_path)
    
    # 2. Reorient to RAS
    img = sitk.DICOMOrient(img, "RAS")
    
    # 3. Resample
    original_spacing = img.GetSpacing()
    original_size = img.GetSize()
    new_size = [int(round(osz * ospc / tspc)) for osz, ospc, tspc in zip(original_size, original_spacing, target_spacing)]
    
    resampler = sitk.ResampleImageFilter()
    resampler.SetOutputSpacing(target_spacing)
    resampler.SetSize(new_size)
    resampler.SetOutputOrigin(img.GetOrigin())
    resampler.SetOutputDirection(img.GetDirection())
    resampler.SetInterpolator(sitk.sitkNearestNeighbor if is_mask else sitk.sitkBSpline)
    img = resampler.Execute(img)
    
    if not is_mask:
        # 4. Simple Skull Strip (Otsu-based)
        mask = sitk.OtsuThreshold(img, 0, 1)
        
        # 5. N4 Bias Correction
        corrector = sitk.N4BiasFieldCorrectionImageFilter()
        img = corrector.Execute(img, mask)
        
        # 6. Z-Score Normalization
        array = sitk.GetArrayFromImage(img)
        mean, std = np.mean(array), np.std(array)
        array = (array - mean) / (std + 1e-8)
        img = sitk.GetImageFromArray(array)
        
    return img

print("✅ Preprocessing logic implemented.")

## 🧪 1.4 Architecture: 3D Residual U-Net
We implement a 3D Residual U-Net using MONAI, optimized for small batch sizes with InstanceNorm.

In [ ]:
from monai.networks.nets import UNet
import torch

def get_model(in_channels=1, out_channels=1):
    model = UNet(
        spatial_dims=3,
        in_channels=in_channels,
        out_channels=out_channels,
        channels=(16, 32, 64, 128, 256),
        strides=(2, 2, 2, 2),
        num_res_units=2,
        norm="INSTANCE",
        act="LEAKYRELU",
        dropout=0.1
    )
    return model

print("✅ 3D U-Net Architecture loaded.")